# 08. 전주 라벨 역추론 디벨롭

팀원 방식(요약): HGB(5시드)+LogReg 순위평균, 피처 23 + 전주라벨 역추론 8 / 학습: 07-26~08-05 재현쌍(D: FUT26 라벨, D−7: FUT18 라벨) / 예측: 09-01 피처 + 공식 08-25 라벨 + 08-25~08-31 기지 구간 / 재현검증 0.559.

**핵심 아이디어**: 공식 08-25 라벨 = "08-25~09-12 영상 중앙값이 임계 이상인가". 이 중 09-01~09-12는 본선 정답 구간(09-01~09-26)의 앞부분 → 정답 구간 자체에 대한 정보.
08-25~08-31 영상(기지 구간 K)은 이미 view_5d를 알기 때문에, 라벨과 K를 함께 보면 09-01~09-12 영상(미지 구간 U)이 높았는지 낮았는지 역추론할 수 있다.

이 노트북: (1) 재현쌍 데이터 구축 (2) 역추론 피처 (3) 개선안: 조건부 시뮬레이션·피처 정리·라벨 노이즈 (4) 채널 GroupKFold로 비교 (5) 제출 파일.

In [1]:
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW = '../data/raw/epoch_data'
D = '2026-09-01'          # 본선 기준일
FUT = 26                  # 본선 정답 구간 길이 (일)
SEED = 42

con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

# con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

In [2]:
def make_features(Dref):
    f = con.sql(f"""
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ),
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                        AS n_long_5d,
               MEDIAN(view_5d)                                 AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)     AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)    AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14)   AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))            AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))         AS comment_rate,
               STDDEV(LN(1 + view_5d))                         AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)             AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)             AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)          AS trend_slope,
               MIN(days_ago)                                   AS days_since_meas
        FROM lv GROUP BY 1
    ),
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    """).df()

    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

FEATURES = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
            'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
            'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
            'log_short_med_rel', 'log_n_long_5d']
print(len(FEATURES), 'features')
make_features(D)[['channel_id'] + FEATURES].head()

23 features


,channel_id,log_past_med,n_long_5d,like_rate,n_long_14,n_short_14,short_ratio,days_since_long,mom_14,log_view_std,n_games,has_game,mom_7,mom_old,comment_rate,log_q10,log_q90,trend_slope,days_since_meas,n_long_7,n_long_28,avg_dur,log_short_med_rel,log_n_long_5d
0,UC-aXNBQ6nwMNeW2V8XB7sBg,8.153925,11,0.019240,3,13,0.8125,2.875000,0.652199,0.666066,1,1,1.769806,0.751920,0.003232,7.843456,9.105202,0.026335,2.875000,1,7,515.000000,3.288632,2.484907
1,UC6ApCfhTtNb1hiUHFffXqvQ,5.418320,2,0.000000,0,0,NaN,44.666667,NaN,0.443592,0,0,NaN,NaN,0.003205,5.118979,5.620846,-0.268857,44.666667,0,0,274.000000,NaN,1.098612
2,UC9otskL_kd-0CDib_5Lj-mQ,10.147277,14,0.006595,2,0,0.0000,0.500000,-0.050218,1.663690,0,0,-0.050218,-0.050218,0.001800,8.975836,12.263632,0.017362,0.500000,2,3,2051.833333,-0.715955,2.708050
3,UC7wosSPUA3os3b3-RMZZe6g,5.114995,8,0.040323,0,0,NaN,14.583333,NaN,2.037604,1,1,NaN,NaN,0.000000,3.149867,6.137694,0.062374,14.583333,0,2,8258.083333,2.675701,2.197225
4,UCLHwMFguUxHVSk91vzicEJw,11.036952,21,0.012020,4,0,0.0000,2.458333,0.457703,0.618171,1,1,1.016436,0.465329,0.001148,10.194103,11.440570,0.019800,2.458333,1,9,1014.461538,-0.628788,3.091042


In [3]:
def logreg(C=0.1):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=2000, C=C, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)

def blend(proba, df):
    """모델 확률 순위 + 과거 중앙값 역순위의 평균 (PR-AUC는 순위만 봄)"""
    return (rank01(proba) + rank01(-df.log_past_med)) / 2



In [4]:
def recent_feats(Dref):
    return con.sql(f"""
    WITH lv AS (
        SELECT channel_id, LN(1 + view_5d) AS l,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago,
               ROW_NUMBER() OVER (PARTITION BY channel_id ORDER BY published_at DESC) AS rn
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ), med AS (SELECT channel_id, MEDIAN(l) AS m FROM lv GROUP BY 1)
    SELECT lv.channel_id,
           SUM(l * EXP(-days_ago / 3.0))  / SUM(EXP(-days_ago / 3.0))  AS ew3,
           SUM(l * EXP(-days_ago / 7.0))  / SUM(EXP(-days_ago / 7.0))  AS ew7,
           SUM(l * EXP(-days_ago / 14.0)) / SUM(EXP(-days_ago / 14.0)) AS ew14,
           MAX(l) FILTER (WHERE rn = 1) AS last1,
           MEDIAN(l) FILTER (WHERE rn <= 3) AS last3,
           AVG(CASE WHEN l > med.m THEN 1.0 ELSE 0.0 END) FILTER (WHERE rn <= 5) AS hot5
    FROM lv JOIN med USING (channel_id) GROUP BY 1
    """).df()



## 1. 재현쌍 데이터
- `labels_full(D, fut)`: 공식 규칙 재현 + growth·과거 수준·임계값까지 반환
- 기준일 D(07-26~08-05)마다: 정답 = D의 FUT26 라벨, 전주 라벨 = (D−7)의 FUT18 라벨
- 기지 구간 K = [D−7, D) 롱폼의 view_5d (09-01에서는 08-25~08-31, 전부 view_5d 있음 — 데이터로 확인)
- 9/1 예측 때는 08-25 임계값 t를 모르므로, 재현 날짜들의 t가 안정적인지 확인하고 평균으로 추정한다.

In [5]:
from datetime import datetime, timedelta
shift = lambda d, k: (datetime.strptime(d, '%Y-%m-%d') + timedelta(days=k)).strftime('%Y-%m-%d')

def labels_full(Dref, fut):
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COUNT(view_5d)  FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                                           AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY) AS n_fut,
                 COALESCE(MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                                           AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL GROUP BY 1)
        SELECT channel_id, LN(1 + past_score) AS past_l, n_fut,
               LN(1 + future_score) - LN(1 + past_score) AS growth
        FROM agg WHERE n_past >= 3 AND past_score >= 100""").df()
    t = np.quantile(g.growth, 0.8)
    g['target'] = (g.growth >= t).astype(int)
    g['ref_date'] = Dref
    return g, t

def known_window(D0, D1):
    """[D0, D1) 롱폼 view_5d (기지 구간 K) 를 채널별 리스트로"""
    k = con.sql(f"""SELECT channel_id, LIST(LN(1 + view_5d)) AS K
                    FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL
                      AND published_at >= TIMESTAMP '{D0}' AND published_at < TIMESTAMP '{D1}' GROUP BY 1""").df()
    return k

def past_level(Dref):
    return con.sql(f"""SELECT channel_id, LN(1 + MEDIAN(view_5d)) AS past_l_prev
                       FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL
                         AND published_at < TIMESTAMP '{Dref}' GROUP BY 1""").df()

PAIR_DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()
pieces, tinfo = [], []
for D_ in PAIR_DATES:
    cur, t_cur = labels_full(D_, 26)
    prv, t_prv = labels_full(shift(D_, -7), 18)
    tinfo.append((D_, round(t_cur, 4), round(t_prv, 4)))
    df = cur[['channel_id', 'ref_date', 'target', 'past_l']].merge(
        prv[['channel_id', 'target']].rename(columns={'target': 'prev'}), on='channel_id', how='left')
    df = (df.merge(past_level(shift(D_, -7)), on='channel_id', how='left')
            .merge(known_window(shift(D_, -7), D_), on='channel_id', how='left')
            .merge(make_features(D_), on='channel_id', how='left')
            .merge(recent_feats(D_), on='channel_id', how='left'))
    df['t_prev_true'] = t_prv
    pieces.append(df)
pairs = pd.concat(pieces, ignore_index=True)
pairs['ew3_d'] = pairs.ew3 - pairs.log_past_med
tinfo = pd.DataFrame(tinfo, columns=['D', 't_cur(FUT26)', 't_prev(D-7, FUT18)'])
print(tinfo.to_string())
print('t_cur 평균/표준편차', round(tinfo.iloc[:, 1].mean(), 4), round(tinfo.iloc[:, 1].std(), 4),
      '| t_prev 평균/표준편차', round(tinfo.iloc[:, 2].mean(), 4), round(tinfo.iloc[:, 2].std(), 4))
print('재현쌍', len(pairs), '행 | 전주 라벨 있는 비율', round(pairs.prev.notna().mean(), 3))
print('전주 라벨별 Rising 비율:', pairs.groupby(pairs.prev.fillna(-1)).target.mean().round(3).to_dict())

             D  t_cur(FUT26)  t_prev(D-7, FUT18)
0   2026-07-26        0.2650              0.2645
1   2026-07-27        0.2445              0.2558
2   2026-07-28        0.2837              0.2268
3   2026-07-29        0.2905              0.2343
4   2026-07-30        0.2670              0.2766
5   2026-07-31        0.2700              0.2752
6   2026-08-01        0.2727              0.2823
7   2026-08-02        0.3105              0.3042
8   2026-08-03        0.3874              0.2517
9   2026-08-04        0.4237              0.2737
10  2026-08-05        0.4600              0.2515
t_cur 평균/표준편차 0.3159 0.073 | t_prev 평균/표준편차 0.2633 0.0223
재현쌍 5102 행 | 전주 라벨 있는 비율 0.569
전주 라벨별 Rising 비율: {-1.0: 0.217, 0.0: 0.109, 1.0: 0.503}


### 1.1 관찰
- 재현쌍 5,102행 (팀원 요약과 일치). 전주 라벨=1 → Rising 0.503, =0 → 0.109.
- 전주 임계값 t_prev = 0.263 ± 0.022로 안정 → 9/1에서는 평균으로 추정 (학습·예측 모두 같은 추정값 사용).
- 학습쌍은 전주 라벨 보유 57%뿐, 본선은 96%(666/694) → 검증을 **전주 라벨 있는 행**에서도 따로 본다.

## 2. 역추론 피처 + 조건부 시뮬레이션
**역추론 피처**: 전주 라벨=1 ⇔ median(K ∪ U) ≥ c, c = 전주 과거수준 + t_prev.
- `margin` = K 중앙값 − c, `pos_low` = 라벨1인데 K가 c보다 낮은 정도(→ U가 높았음), `neg_high` = 라벨0인데 K가 높은 정도(→ U가 낮았음)

**조건부 시뮬레이션**: 채널의 최근 영상 분포에서 U(=D~D+11일 영상)를 반복 생성 → 전주 라벨과 일치하는 경우만 남김(기각 샘플링) → 여기에 V(D+11~D+26일)를 더해 D의 growth ≥ t_cur 확률 계산.

In [6]:
T_PREV = float(tinfo.iloc[:, 2].mean())
T_CUR  = float(tinfo.iloc[:8, 1].mean())   # 08-03 이후는 t가 튀어 앞 8일 평균 사용
print('T_PREV', round(T_PREV, 4), '| T_CUR', round(T_CUR, 4))

def inv_feats(df, t_prev=T_PREV):
    out = pd.DataFrame(index=df.index)
    K = df.K.apply(lambda x: np.asarray(x, float) if isinstance(x, (list, np.ndarray)) else np.array([]))
    kn = K.apply(len); kmed = K.apply(lambda a: np.median(a) if len(a) else np.nan)
    c = df.past_l_prev + t_prev
    margin = kmed - c
    prev = df.prev
    out['prev_f'] = prev.fillna(0.2)
    out['prev_avail'] = prev.notna().astype(int)
    out['K_n'] = np.log1p(kn)
    out['margin'] = margin.fillna(-1.0)
    out['pos_low'] = (prev == 1) * np.clip(-margin.fillna(-1.0), 0, None)
    out['neg_high'] = (prev == 0) * np.clip(margin.fillna(-1.0), 0, None)
    out['prev_margin'] = out.prev_f * out.margin
    out['K_med_d'] = (kmed - df.log_past_med).fillna(0)
    return out
INV = ['prev_f', 'prev_avail', 'K_n', 'margin', 'pos_low', 'neg_high', 'prev_margin', 'K_med_d']

def chan_dist(Dref, days=28):
    return con.sql(f"""SELECT channel_id,
                              LIST(LN(1 + view_5d)) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL {days} DAY) AS Lr,
                              LIST(LN(1 + view_5d)) AS La
                       FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL
                         AND published_at < TIMESTAMP '{Dref}' GROUP BY 1""").df()

def _nanmed_rows(a):
    m = np.nanmedian(np.where(np.isnan(a).all(axis=1, keepdims=True), 0.0, a), axis=1)
    return m

def cond_sim(df, dists, t_prev=T_PREV, t_cur=T_CUR, n=400, jitter=0.3, u_days=11, v_days=15, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    sc_c, sc_p, upost = np.zeros(len(df)), np.zeros(len(df)), np.full(len(df), np.nan)
    for j, (r, Lr, La) in enumerate(zip(df.itertuples(index=False), dd.Lr, dd.La)):
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0:
            continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku, kv = rng.poisson(rate * u_days, n), rng.poisson(rate * v_days, n)
        mu_, mv_ = max(ku.max(), 1), max(kv.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        V = rng.choice(L, (n, mv_)) + rng.normal(0, jitter, (n, mv_)); V[np.arange(mv_)[None, :] >= kv[:, None]] = np.nan
        fut = _nanmed_rows(np.hstack([U, V]))
        hit = (fut - r.past_l) >= t_cur
        sc_p[j] = hit.mean()
        w = np.ones(n, bool)
        if pd.notna(r.prev) and pd.notna(r.past_l_prev):
            K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
            KU = np.hstack([np.tile(K, (n, 1)), U]) if len(K) else U
            gp = _nanmed_rows(KU) - r.past_l_prev
            w = (gp >= t_prev) == bool(r.prev)
            if w.sum() < 5:
                w = np.ones(n, bool)
        sc_c[j] = hit[w].mean()
        um = _nanmed_rows(U)[w]
        upost[j] = np.mean(um[~np.isnan(um)]) - r.past_l if len(um) else np.nan
    return sc_c, sc_p, upost

pairs = pairs.join(inv_feats(pairs))
sims = []
for D_ in PAIR_DATES:
    m = pairs.ref_date == D_
    c_, p_, u_ = cond_sim(pairs[m], chan_dist(D_))
    sims.append(pd.DataFrame({'sim_cond': c_, 'sim_prior': p_, 'u_post': u_}, index=pairs.index[m]))
pairs = pairs.join(pd.concat(sims))
for k in ['sim_cond', 'sim_prior']:
    per = [average_precision_score(g.target, g[k]) for _, g in pairs.groupby('ref_date')]
    per_p = [average_precision_score(g.target, g[k]) for _, g in pairs[pairs.prev.notna()].groupby('ref_date')]
    print(f'{k:10s} 단독  날짜평균 {np.mean(per):.4f} | 전주라벨 있는 행 {np.mean(per_p):.4f}')

T_PREV 0.2633 | T_CUR 0.2755


sim_cond   단독  날짜평균 0.4472 | 전주라벨 있는 행 0.4965
sim_prior  단독  날짜평균 0.3516 | 전주라벨 있는 행 0.3356


### 2.1 시뮬레이션 단독 (학습 없음)
- 사전 시뮬레이션 0.347 → **조건부 시뮬레이션 0.440** (전주 라벨 있는 행 0.331 → 0.486). 전주 라벨로 걸러내는 것만으로 큰 폭 상승.

## 3. 모델 비교 (채널 GroupKFold 5-fold, 11개 기준일, 시드 0·1 평균)
- `전체`: 날짜별 PR-AUC 평균 / `전주있음`: 전주 라벨 있는 행만 (본선 96%가 여기에 해당)
- M0: 05 방식 (LR23 C=0.1 + rule 0.5) — 전주 라벨 없음
- M1: **팀원 방식 재현** — HGB(5시드)+LR 순위평균, 23+역추론8
- M2: M1 + 조건부 시뮬레이션 피처
- M3: PRUNED+ew3 + 역추론8 + 시뮬 피처, LR C=0.01
- M4: M3 피처로 HGB+LR
- M5: M4 + 조건부 시뮬레이션 점수 순위 블렌드

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
DROP = ['comment_rate', 'avg_dur', 'log_q90', 'like_rate', 'log_q10', 'trend_slope', 'has_game']
PRUNED = [f for f in FEATURES if f not in DROP]
SIMF = ['sim_cond', 'u_post']

def hgb(seed):
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.03, max_iter=300, min_samples_leaf=40,
                                          l2_regularization=1.0, random_state=seed)
def p_lr(trn, tst, cols, C=0.1):
    return logreg(C).fit(trn[cols], trn.target).predict_proba(tst[cols])[:, 1]
def p_hgb(trn, tst, cols, seeds=5):
    return np.mean([hgb(s).fit(trn[cols], trn.target).predict_proba(tst[cols])[:, 1] for s in range(seeds)], axis=0)

def run_cv(df, comp_fns, seed=0, k=5):
    ch = df.channel_id.values; u = np.unique(ch); rg = np.random.default_rng(seed); rg.shuffle(u)
    fo = {c: i % k for i, c in enumerate(u)}; fold = np.array([fo[c] for c in ch])
    out = {n: np.zeros(len(df)) for n in comp_fns}
    for f in range(k):
        trn, tst = df[fold != f], df[fold == f]
        for n, fn in comp_fns.items():
            out[n][fold == f] = fn(trn, tst)
    return pd.DataFrame(out, index=df.index)

def summ(df, sc):
    R = {}
    for c in sc.columns:
        a = [average_precision_score(g.target, sc.loc[g.index, c]) for _, g in df.groupby('ref_date')]
        b = [average_precision_score(g.target, sc.loc[g.index, c]) for _, g in df[df.prev.notna()].groupby('ref_date')]
        R[c] = {'전체': np.mean(a), '전주있음': np.mean(b)}
    return pd.DataFrame(R).T

F23I  = FEATURES + INV
F23IS = FEATURES + INV + SIMF
FPIS  = PRUNED + ['ew3_d'] + INV + SIMF
comps = {
    'lr23':    lambda a, b: p_lr(a, b, FEATURES),
    'lr23i':   lambda a, b: p_lr(a, b, F23I),
    'hgb23i':  lambda a, b: p_hgb(a, b, F23I),
    'lr23is':  lambda a, b: p_lr(a, b, F23IS),
    'hgb23is': lambda a, b: p_hgb(a, b, F23IS),
    'lrP':     lambda a, b: p_lr(a, b, FPIS, C=0.01),
    'hgbP':    lambda a, b: p_hgb(a, b, FPIS),
    'rule0':   lambda a, b: (-b.log_past_med).values,
    'simc':    lambda a, b: b.sim_cond.values,
}
res = []
for seed in [0, 1]:
    raw = run_cv(pairs, comps, seed=seed)
    R = lambda c: raw[c].groupby(pairs.ref_date).rank(pct=True)
    sc = pd.DataFrame({
        'M0 05방식 (LR23+rule)':        0.5 * R('lr23') + 0.5 * R('rule0'),
        'M1 팀원방식 (HGB+LR, 23+역추론)': 0.5 * R('hgb23i') + 0.5 * R('lr23i'),
        'M2 M1 + 시뮬 피처':             0.5 * R('hgb23is') + 0.5 * R('lr23is'),
        'M3 LR(PRUNED+역추론+시뮬)':      R('lrP'),
        'M4 HGB+LR(PRUNED+역추론+시뮬)':  0.5 * R('hgbP') + 0.5 * R('lrP'),
        'M5 M4 + 조건부시뮬 0.3':         0.35 * R('hgbP') + 0.35 * R('lrP') + 0.3 * R('simc'),
        'S  조건부시뮬 단독':              R('simc'),
    }, index=pairs.index)
    res.append(summ(pairs, sc))
cmp8 = pd.concat(res).groupby(level=0).mean().round(4).sort_values('전주있음', ascending=False)
cmp8

,전체,전주있음
M5 M4 + 조건부시뮬 0.3,0.4939,0.5555
M4 HGB+LR(PRUNED+역추론+시뮬),0.4963,0.5549
M3 LR(PRUNED+역추론+시뮬),0.4999,0.5529
M2 M1 + 시뮬 피처,0.4838,0.5415
"M1 팀원방식 (HGB+LR, 23+역추론)",0.4633,0.5297
S 조건부시뮬 단독,0.4472,0.4965
M0 05방식 (LR23+rule),0.3380,0.3323


### 3.1 결과
| 방법 | 전체 | 전주있음 |
|---|---|---|
| M0 05방식 | 0.338 | 0.332 |
| M1 팀원방식 재현 | 0.463 | 0.530 |
| M2 +시뮬 피처 | 0.480 | 0.536 |
| M3 LR(PRUNED+역추론+시뮬, C=0.01) | **0.500** | 0.548 |
| M4 HGB+LR(M3 피처) | 0.496 | **0.551** |

## 4. 09-01 제출 파일 (M4, M3)
- 전주 라벨 = 공식 08-25 라벨, 전주 과거수준 = 08-25 이전 중앙값, K = 08-25~08-31 롱폼 view_5d
- t_prev = 재현 날짜 평균(0.263), U = 09-01~09-11(11일), V = 이후 15일
- 최종 학습: 재현쌍 전체 5,102행

In [8]:
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
prev825 = tl[tl.row_id.str.endswith('2026-08-25')][['channel_id', 'target']].rename(columns={'target': 'prev'})

test = (sample[['row_id', 'channel_id']]
        .merge(past_level(D).rename(columns={'past_l_prev': 'past_l'}), on='channel_id', how='left')
        .merge(prev825, on='channel_id', how='left')
        .merge(past_level('2026-08-25'), on='channel_id', how='left')
        .merge(known_window('2026-08-25', D), on='channel_id', how='left')
        .merge(make_features(D), on='channel_id', how='left')
        .merge(recent_feats(D), on='channel_id', how='left'))
test['ew3_d'] = test.ew3 - test.log_past_med
test = test.join(inv_feats(test))
c_, p_, u_ = cond_sim(test, chan_dist(D))
test['sim_cond'], test['sim_prior'], test['u_post'] = c_, p_, u_
print('행', len(test), '| 전주 라벨 있음', test.prev.notna().sum(), '| K 있음', test.K.notna().sum(),
      '| past_l 결측', test.past_l.isna().sum())

lr_fin = logreg(0.01).fit(pairs[FPIS], pairs.target)
p_lrP = lr_fin.predict_proba(test[FPIS])[:, 1]
p_hgbP = np.mean([hgb(s).fit(pairs[FPIS], pairs.target).predict_proba(test[FPIS])[:, 1] for s in range(5)], axis=0)

def save_sub(score, fname):
    s = np.asarray(score, float) + 1e-6 * rank01(p_lrP)          # 동점 깨기
    s = (s - s.min()) / (s.max() - s.min())
    sub = sample[['row_id']].merge(pd.DataFrame({'row_id': test.row_id, 'prediction': s}), on='row_id', how='left')
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694 and sub.row_id.is_unique
    assert set(sub.row_id) == set(sample.row_id)
    assert sub.prediction.notna().all() and np.isfinite(sub.prediction).all() and sub.prediction.between(0, 1).all()
    sub.to_csv('../submissions/' + fname, index=False, encoding='utf-8')
    return sub.set_index('row_id').prediction

s4 = save_sub(0.5 * rank01(p_hgbP) + 0.5 * rank01(p_lrP), 'submit_13_M4_previnv_hgb_lr_sim.csv')
s3 = save_sub(rank01(p_lrP), 'submit_14_M3_previnv_lr_sim.csv')
ref = pd.read_csv('../submissions/submit_04_logreg_rankblend_fut26.csv').set_index('row_id').prediction
print('M4 vs submit_04 순위상관', round(s4.corr(ref.loc[s4.index], method='spearman'), 4),
      '| M3 vs submit_04', round(s3.corr(ref.loc[s3.index], method='spearman'), 4),
      '| M4 vs M3', round(s4.corr(s3, method='spearman'), 4))
print('전주 라벨별 평균 예측 순위(M4):', pd.Series(rank01(s4.loc[test.row_id].values)).groupby(test.prev.fillna(-1).values).mean().round(3).to_dict())

행 694 | 전주 라벨 있음 666 | K 있음 598 | past_l 결측 0


M4 vs submit_04 순위상관 0.6481 | M3 vs submit_04 0.6608 | M4 vs M3 0.9539
전주 라벨별 평균 예측 순위(M4): {-1.0: 0.764, 0.0: 0.391, 1.0: 0.883}


### 4.1 결과: submit_13(M4), submit_14(M3) 생성. submit_04와 순위상관 0.65 (완전히 다른 예측). 8/25 라벨 없는 신규 28채널의 평균 순위 0.77 — 5절에서 점검.

## 5. 개선 1: 08-25 임계값 t_prev를 공식 라벨로 추정 (최대우도)
지금은 재현 날짜 평균 t(0.263±0.022)를 쓴다. t가 틀리면 모든 채널의 역추론 기준선이 한꺼번에 밀린다.
방법: 채널마다 U를 시뮬레이션해 `gp = median(K∪U) − 전주 과거수준` 분포를 만들고, t 후보마다 `Σ log P(전주 라벨 | t)`가 최대인 t를 고른다.
**먼저 정답 t를 아는 재현 날짜에서 추정 오차를 확인**한 뒤 08-25에 적용한다.

In [9]:
def sim_gp(df, dists, n=400, jitter=0.3, u_days=11, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    out = []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        if pd.isna(r.prev) or pd.isna(r.past_l_prev):
            out.append(None); continue
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0:
            out.append(None); continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku = rng.poisson(rate * u_days, n); mu_ = max(ku.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
        KU = np.hstack([np.tile(K, (n, 1)), U]) if len(K) else U
        out.append(_nanmed_rows(KU) - r.past_l_prev)
    return out

T_GRID = np.round(np.arange(0.10, 0.45, 0.005), 3)
def mle_t(df, gps):
    y = df.prev.values
    ll = []
    for t in T_GRID:
        s = 0.0
        for yi, g in zip(y, gps):
            if g is None: continue
            p = np.clip((g >= t).mean(), 0.01, 0.99)
            s += np.log(p if yi == 1 else 1 - p)
        ll.append(s)
    return float(T_GRID[int(np.argmax(ll))])

rows = []
for D_ in PAIR_DATES:
    d = pairs[pairs.ref_date == D_]
    gps = sim_gp(d, chan_dist(D_))
    rows.append((D_, round(d.t_prev_true.iloc[0], 4), mle_t(d, gps)))
tchk = pd.DataFrame(rows, columns=['D', 't_prev 정답', 't_prev MLE'])
tchk['MLE 오차'] = (tchk['t_prev MLE'] - tchk['t_prev 정답']).round(4)
tchk['평균값 오차'] = (T_PREV - tchk['t_prev 정답']).round(4)
print(tchk.to_string())
print('평균 절대오차  MLE:', round(tchk['MLE 오차'].abs().mean(), 4), '| 평균값:', round(tchk['평균값 오차'].abs().mean(), 4))
print('MLE 편향(평균 오차):', round(tchk['MLE 오차'].mean(), 4))

             D  t_prev 정답  t_prev MLE  MLE 오차  평균값 오차
0   2026-07-26     0.2645       0.220 -0.0445 -0.0012
1   2026-07-27     0.2558       0.255 -0.0008  0.0075
2   2026-07-28     0.2268       0.235  0.0082  0.0365
3   2026-07-29     0.2343       0.235  0.0007  0.0290
4   2026-07-30     0.2766       0.260 -0.0166 -0.0133
5   2026-07-31     0.2752       0.245 -0.0302 -0.0119
6   2026-08-01     0.2823       0.250 -0.0323 -0.0190
7   2026-08-02     0.3042       0.305  0.0008 -0.0409
8   2026-08-03     0.2517       0.255  0.0033  0.0116
9   2026-08-04     0.2737       0.265 -0.0087 -0.0104
10  2026-08-05     0.2515       0.255  0.0035  0.0118
평균 절대오차  MLE: 0.0136 | 평균값: 0.0176
MLE 편향(평균 오차): -0.0106


### 5.1 결과: MLE 평균 절대오차 0.0128 < 평균값 0.0176, 편향 −0.007 (보정해서 사용).
### 5.2 t 정확도가 점수에 미치는 영향 (평균값 / 날짜별 MLE / 정답 t) + 08-25 t 추정

In [10]:
MLE_BIAS = float(tchk['MLE 오차'].mean())
t_by = {'평균값': {d: T_PREV for d in PAIR_DATES},
        'MLE(보정)': dict(zip(tchk.D, tchk['t_prev MLE'] - MLE_BIAS)),
        '정답': dict(zip(tchk.D, tchk['t_prev 정답']))}
base_cols = [c for c in pairs.columns if c not in INV + ['sim_cond', 'sim_prior', 'u_post']]
variants = {}
for name, tmap in t_by.items():
    parts = []
    for D_ in PAIR_DATES:
        d = pairs.loc[pairs.ref_date == D_, base_cols].copy()
        d = d.join(inv_feats(d, t_prev=tmap[D_]))
        c_, p_, u_ = cond_sim(d, chan_dist(D_), t_prev=tmap[D_])
        d['sim_cond'], d['sim_prior'], d['u_post'] = c_, p_, u_
        parts.append(d)
    variants[name] = pd.concat(parts)
rows = []
for name, dfv in variants.items():
    for seed in [0, 1]:
        raw = run_cv(dfv, {'lrP': lambda a, b: p_lr(a, b, FPIS, C=0.01)}, seed=seed)
        s = summ(dfv, raw); s.index = [name]; rows.append(s)
print(pd.concat(rows).groupby(level=0).mean().round(4))

# 08-25 t 추정 (테스트 데이터, 08-25 이전 시점 분포로 U 생성은 09-01 분포 사용 — 학습쌍과 동일하게 D 시점 분포)
gps_t = sim_gp(test, chan_dist(D))
t825_raw = mle_t(test, gps_t)
T825 = t825_raw - MLE_BIAS
print('08-25 t MLE:', t825_raw, '→ 편향 보정', round(T825, 4), '(기존 평균값', round(T_PREV, 4), ')')

             전체    전주있음
MLE(보정)  0.4989  0.5503
정답       0.5016  0.5563
평균값      0.4999  0.5529


08-25 t MLE: 0.445 → 편향 보정 0.4556 (기존 평균값 0.2633 )


### 5.3 관찰: 학습 날짜에선 t 정확도 영향이 작음(0.5532→0.5553). 그러나 **08-25의 MLE t = 0.445(탐색 범위 끝)** 로 평균값 0.263과 크게 다름.
FUT26 정답 임계값도 정답 구간이 8월 말로 끝나는 08-03~08-05에서 0.39~0.46으로 튐 → **8월 말 영상의 view_5d가 체계적으로 높게 측정됐을 가능성**.
그렇다면 기지 구간 K(08-25~08-31)가 부풀려져, t=0.263으로 역추론하면 "K 높은데 라벨0 → U 낮음" 오추론이 생긴다 (CV로는 안 보이는 테스트 전용 문제).

### 5.4 확인: (a) 탐색 범위 확장 (b) 업로드일별 view_5d 측정 시점·수준 (c) 학습/테스트 역추론 피처 분포 비교

In [11]:
T_GRID = np.round(np.arange(0.10, 0.90, 0.01), 3)
t825_raw = mle_t(test, gps_t); T825 = t825_raw - MLE_BIAS
print('(a) 08-25 t MLE (범위 확장):', t825_raw, '→ 보정', round(T825, 4))

print('(b) 업로드일별 view_5d 측정 시점(h_before/h_after)과 채널 내 상대 수준')
q = con.sql("""
    WITH v AS (SELECT channel_id, published_at, h_before, h_after, LN(1 + view_5d) l
               FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL),
    cm AS (SELECT channel_id, MEDIAN(l) m FROM v WHERE published_at < TIMESTAMP '2026-08-15' GROUP BY 1)
    SELECT DATE_TRUNC('day', published_at) d, COUNT(*) n,
           ROUND(MEDIAN(h_before), 1) h_before, ROUND(MEDIAN(h_after), 1) h_after,
           ROUND(MEDIAN(l - cm.m), 3) rel_level
    FROM v JOIN cm USING (channel_id)
    WHERE published_at >= TIMESTAMP '2026-08-10' GROUP BY 1 ORDER BY 1""").df()
print(q.to_string())

print('(c) 라벨0인데 K가 기준선보다 높은(neg_high>0) 비율 / 라벨1인데 K가 낮은(pos_low>0) 비율')
def share(df, t):
    f = inv_feats(df, t_prev=t)
    p = df.prev
    return round(((f.neg_high > 0) & (p == 0)).sum() / (p == 0).sum(), 3), round(((f.pos_low > 0) & (p == 1)).sum() / (p == 1).sum(), 3)
print('  학습쌍 (날짜별 정답 t):', share(pairs.assign(), T_PREV))
print('  테스트, t=0.263:', share(test, T_PREV))
print('  테스트, t=MLE 보정:', share(test, T825))

(a) 08-25 t MLE (범위 확장): 0.89 → 보정 0.9006
(b) 업로드일별 view_5d 측정 시점(h_before/h_after)과 채널 내 상대 수준
            d    n  h_before  h_after  rel_level
0  2026-08-10  445     120.0    120.1      0.000
1  2026-08-11  419     120.0    120.1     -0.033
2  2026-08-12  485     120.0    120.1      0.000
3  2026-08-13  511     120.0    120.1     -0.141
4  2026-08-14  521     120.0    120.1     -0.008
5  2026-08-15  506     120.0    120.1     -0.056
6  2026-08-16  431     120.0    120.1      0.062
7  2026-08-17  412     120.0    120.1      0.023
8  2026-08-18  434     119.9    120.0      0.000
9  2026-08-19  475     119.9    120.0     -0.043
10 2026-08-20  476     120.0    120.0     -0.031
11 2026-08-21  488     119.8    120.0      0.020
12 2026-08-22  494     119.9    120.0      0.015
13 2026-08-23  445     119.9    120.1      0.050
14 2026-08-24  445     119.9    120.0      0.084
15 2026-08-25  477     119.8    120.0      0.125
16 2026-08-26  488     119.8    120.0      0.431
17 2026-08-27  504    

### 5.5 발견: **08-26 이후 업로드 영상의 view_5d가 채널 자기 수준보다 +0.43~0.85(로그) 높다** (08-25 이전은 ±0.1).
측정 시점 컬럼(h_before/h_after)은 정상 120h. 원인(실제 급등/측정 차이)은 불명.
영향: 09-01의 최근성 피처(mom_7, ew3 등)가 학습 날짜엔 없던 방식으로 부풀려지고, 08-26~31에 업로드한 채널만 과대평가됨. 역추론도 t=0.263이면 라벨0 채널의 59%가 "K 높음→U 낮음"으로 오추론(학습쌍은 9%).

### 5.6 급등이 채널 공통 효과인가? — 업로드일 효과 δ(day) 추정 (log view = 채널효과 + 일효과)

In [12]:
vv = con.sql("""SELECT channel_id, CAST(DATE_TRUNC('day', published_at) AS DATE) AS day, is_shorts, LN(1 + view_5d) AS l
                FROM video_5d_views WHERE view_5d IS NOT NULL""").df()
def day_effect(v, iters=8):
    v = v.copy(); v['dlt'] = 0.0
    for _ in range(iters):
        a = (v.l - v.dlt).groupby(v.channel_id).median()
        v['a'] = v.channel_id.map(a)
        dl = (v.l - v.a).groupby(v.day).median()
        dl = dl - dl[dl.index < pd.Timestamp('2026-08-20')].median()   # 08-20 이전 기준 0
        v['dlt'] = v.day.map(dl)
    return dl, v
dl_long, vl = day_effect(vv[vv.is_shorts == 0])
dl_short, vs_ = day_effect(vv[vv.is_shorts == 1])
eff = pd.DataFrame({'롱폼 δ': dl_long.round(3), '쇼츠 δ': dl_short.round(3)})
print(eff[eff.index >= pd.Timestamp('2026-08-15')].to_string())
print('07-10~08-20 롱폼 δ 범위:', round(dl_long[dl_long.index < pd.Timestamp('2026-08-20')].min(), 3), '~',
      round(dl_long[dl_long.index < pd.Timestamp('2026-08-20')].max(), 3))
# 공통 효과인지: 08-27~08-31 영상의 (l - 채널효과) 분포가 좁게 몰려 있는가, 채널 대부분이 올랐는가
late = vl[vl.day >= pd.Timestamp('2026-08-27')]
r_ = (late.l - late.a)
print('08-27~31 롱폼 상대수준 분위수 10/25/50/75/90%:', np.round(np.quantile(r_, [0.1, 0.25, 0.5, 0.75, 0.9]), 3))
ch_up = (late.assign(r=r_).groupby('channel_id').r.median() > 0.3).mean()
print('08-27~31 업로드 채널 중 자기 수준보다 +0.3 이상 오른 채널 비율:', round(ch_up, 3))
early = vl[(vl.day >= pd.Timestamp('2026-08-10')) & (vl.day < pd.Timestamp('2026-08-20'))]
print('비교: 08-10~19 같은 비율:', round((early.assign(r=early.l - early.a).groupby('channel_id').r.median() > 0.3).mean(), 3))

             롱폼 δ   쇼츠 δ
day                     
2026-08-15 -0.042 -0.067
2026-08-16  0.018 -0.074
2026-08-17  0.018 -0.043
2026-08-18 -0.033 -0.019
2026-08-19 -0.117 -0.142
2026-08-20 -0.067  0.000
2026-08-21 -0.009 -0.038
2026-08-22 -0.022  0.000
2026-08-23  0.040 -0.002
2026-08-24  0.045 -0.007
2026-08-25  0.092 -0.021
2026-08-26  0.326  0.029
2026-08-27  0.661  0.058
2026-08-28  0.759  0.058
2026-08-29  0.804  0.106
2026-08-30  0.789 -0.016
2026-08-31  0.782  0.000
07-10~08-20 롱폼 δ 범위: -0.158 ~ 0.097
08-27~31 롱폼 상대수준 분위수 10/25/50/75/90%: [-0.195  0.346  0.78   1.312  2.041]
08-27~31 업로드 채널 중 자기 수준보다 +0.3 이상 오른 채널 비율: 0.816
비교: 08-10~19 같은 비율: 0.147


### 5.7 결과: **롱폼에만 생긴 채널 공통 급등**
- 업로드일 효과 δ: 08-26 +0.33, 08-27~31 +0.66~0.80 (07-10~08-20은 −0.16~+0.10). 쇼츠는 정상.
- 08-27~31 업로드 채널의 82%가 자기 수준 +0.3 이상 (평소 15%) → 특정 채널이 아닌 데이터 전체 효과.

## 6. 급등 보정
1. **피처**: 08-24 이후 롱폼 view_5d에서 δ(day)를 빼고 최근성 피처 재계산. `log_past_med`·`past_l`(라벨과 정확히 일치해야 함)은 원값 유지. 학습 날짜는 δ≈0이라 영향 없음.
2. **역추론**: 9월 영상(U)의 급등 크기 s_U를 공식 08-25 라벨로 최대우도 추정 (t는 평소 값 0.263 고정). 학습 날짜에선 s_U≈0이 나와야 정상.
3. **시뮬 임계값**: 고정 T_CUR 대신 채널 전체 시뮬 growth의 80% 분위수로 계산 (급등에도 자동 보정).

In [13]:
# ---- 1. δ(day) 적용 뷰: 08-24 이후 롱폼만 보정 ----
DELTA = dl_long[dl_long.index >= pd.Timestamp('2026-08-24')].clip(lower=0)
delta_df = pd.DataFrame({'day': pd.to_datetime(DELTA.index), 'dlt': DELTA.values})
con.register('delta_df', delta_df)
con.sql(f"CREATE OR REPLACE VIEW v5_raw AS SELECT * FROM read_csv('{RAW}/video_5d_views.csv', header=true, auto_detect=true)")
def use_adjusted(flag):
    if flag:
        con.sql("""CREATE OR REPLACE VIEW video_5d_views AS
                   SELECT r.* REPLACE (CASE WHEN r.is_shorts = 0 AND r.view_5d IS NOT NULL AND d.dlt IS NOT NULL
                                            THEN EXP(LN(1 + r.view_5d) - d.dlt) - 1 ELSE r.view_5d END AS view_5d)
                   FROM v5_raw r LEFT JOIN delta_df d ON CAST(DATE_TRUNC('day', r.published_at) AS DATE) = CAST(d.day AS DATE)""")
    else:
        con.sql("CREATE OR REPLACE VIEW video_5d_views AS SELECT * FROM v5_raw")
use_adjusted(False)
print(DELTA.round(3).to_dict())

# ---- 2·3. 일반화된 조건부 시뮬레이션 ----
def cond_sim2(df, dists, s_u=0.0, t_prev=T_PREV, n=400, jitter=0.3, u_days=11, v_days=15, seed=0):
    """dists: (보정된) 채널 분포. K는 원값. U,V는 보정 분포 + s_u. 임계값은 시뮬 growth 풀의 80% 분위수"""
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    G, W, UP = [], [], []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0:
            G.append(None); W.append(None); UP.append(np.nan); continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku, kv = rng.poisson(rate * u_days, n), rng.poisson(rate * v_days, n)
        mu_, mv_ = max(ku.max(), 1), max(kv.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)) + s_u; U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        V = rng.choice(L, (n, mv_)) + rng.normal(0, jitter, (n, mv_)) + s_u; V[np.arange(mv_)[None, :] >= kv[:, None]] = np.nan
        g = _nanmed_rows(np.hstack([U, V])) - r.past_l
        w = np.ones(n, bool)
        if pd.notna(r.prev) and pd.notna(r.past_l_prev):
            K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
            KU = np.hstack([np.tile(K, (n, 1)), U]) if len(K) else U
            w = (_nanmed_rows(KU) - r.past_l_prev >= t_prev) == bool(r.prev)
            if w.sum() < 5: w = np.ones(n, bool)
        G.append(g); W.append(w)
        um = _nanmed_rows(U)[w]; UP.append(np.nanmean(um) - s_u - r.past_l if len(um) else np.nan)
    pool = np.concatenate([g[w][np.random.default_rng(1).integers(0, w.sum(), 50)] for g, w in zip(G, W) if g is not None])
    t_cur = np.quantile(pool, 0.8)
    sc = np.array([np.mean(g[w] >= t_cur) if g is not None else np.nan for g, w in zip(G, W)])
    return np.nan_to_num(sc, nan=np.nanmean(sc)), np.array(UP), t_cur

def mle_su(df, dists, grid=np.round(np.arange(-0.3, 1.31, 0.05), 2), t_prev=T_PREV, n=300, jitter=0.3, u_days=11, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    base = []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        if pd.isna(r.prev) or pd.isna(r.past_l_prev): continue
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0: continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku = rng.poisson(rate * u_days, n); mu_ = max(ku.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
        base.append((int(r.prev), K, U, r.past_l_prev))
    ll = []
    for s in grid:
        tot = 0.0
        for y, K, U, pp in base:
            KU = np.hstack([np.tile(K, (U.shape[0], 1)), U + s]) if len(K) else U + s
            p = np.clip(((_nanmed_rows(KU) - pp) >= t_prev).mean(), 0.01, 0.99)
            tot += np.log(p if y == 1 else 1 - p)
        ll.append(tot)
    return float(grid[int(np.argmax(ll))]), pd.Series(ll, index=grid)

# 점검: 학습 날짜에서 s_U ≈ 0 이어야 함 (날짜별 정답 t 사용)
chk = []
for D_ in PAIR_DATES[::2]:
    d = pairs[pairs.ref_date == D_]
    s_hat, _ = mle_su(d, chan_dist(D_), t_prev=float(d.t_prev_true.iloc[0]))
    chk.append((D_, s_hat))
print('학습 날짜 s_U 추정 (0 근처여야 정상):', chk)

# 테스트: 보정 분포로 U 생성, K는 원값
use_adjusted(True); dist_test_adj = chan_dist(D); use_adjusted(False)
S_U, ll_su = mle_su(test, dist_test_adj)
print('09월 영상 급등 크기 s_U 추정:', S_U, '| 로그우도 상위:', ll_su.sort_values(ascending=False).head(3).round(1).to_dict())

{Timestamp('2026-08-24 00:00:00'): 0.045, Timestamp('2026-08-25 00:00:00'): 0.092, Timestamp('2026-08-26 00:00:00'): 0.326, Timestamp('2026-08-27 00:00:00'): 0.661, Timestamp('2026-08-28 00:00:00'): 0.759, Timestamp('2026-08-29 00:00:00'): 0.804, Timestamp('2026-08-30 00:00:00'): 0.789, Timestamp('2026-08-31 00:00:00'): 0.782}


학습 날짜 s_U 추정 (0 근처여야 정상): [('2026-07-26', 0.1), ('2026-07-28', 0.05), ('2026-07-30', 0.1), ('2026-08-01', 0.05), ('2026-08-03', 0.05), ('2026-08-05', -0.0)]


09월 영상 급등 크기 s_U 추정: -0.3 | 로그우도 상위: {-0.3: -308.1, -0.25: -321.0, -0.2: -339.0}


### 6.1 결과
- 학습 날짜 s_U ≈ 0~0.1 (방법 정상)
- **테스트 s_U = −0.3 (범위 끝)**: K를 원값(급등 포함)으로 두면, 공식 라벨과 맞추려면 9월 영상이 비정상적으로 낮아야 함
→ 가설: **공식 라벨은 급등 없는 정상 view_5d로 계산됐고, 배포 파일의 08-26~31 값만 부풀려져 있다.**

### 6.2 가설 검증
(a) 공식 08-11 라벨(정답 구간 08-11~08-29, 급등 영상 08-26~28 포함)을 원값 vs 보정값으로 재현해 일치율 비교
(b) 08-25 역추론을 보정된 K로 다시 추정 → s_U ≈ 0 이면 가설 지지

In [14]:
tl_ = pd.read_csv(f'{RAW}/train_labels.csv')
tl_['channel_id'] = tl_.row_id.str.rsplit('_', n=1).str[0]; tl_['ref_date'] = tl_.row_id.str.rsplit('_', n=1).str[1]
off811 = tl_[tl_.ref_date == '2026-08-11'][['channel_id', 'target']]
for flag in [False, True]:
    use_adjusted(flag)
    g, t_ = labels_full('2026-08-11', 18)
    m_ = off811.merge(g[['channel_id', 'target']], on='channel_id', suffixes=('_off', '_rep'))
    # 급등 영상(08-26~28)이 정답 구간에 있는 채널만 따로
    late = con.sql("""SELECT DISTINCT channel_id FROM v5_raw WHERE is_shorts = 0 AND view_5d IS NOT NULL
                      AND published_at >= TIMESTAMP '2026-08-26' AND published_at < TIMESTAMP '2026-08-29'""").df().channel_id
    ml = m_[m_.channel_id.isin(set(late))]
    print(f"(a) {'보정값' if flag else '원값  '} 재현 → 공식 08-11 일치율 전체 {(m_.target_off == m_.target_rep).mean():.4f} ({len(m_)}행)"
          f" | 급등영상 있는 채널 {(ml.target_off == ml.target_rep).mean():.4f} ({len(ml)}행)")
use_adjusted(False)

test_adjK = test.copy()
use_adjusted(True)
test_adjK = test_adjK.drop(columns=['K']).merge(known_window('2026-08-25', D), on='channel_id', how='left')
use_adjusted(False)
S_U2, ll2 = mle_su(test_adjK, dist_test_adj)
print('(b) 보정 K로 추정한 s_U:', S_U2, '| 로그우도 상위:', ll2.sort_values(ascending=False).head(3).round(1).to_dict())

(a) 원값   재현 → 공식 08-11 일치율 전체 0.9849 (596행) | 급등영상 있는 채널 0.9887 (441행)


(a) 보정값 재현 → 공식 08-11 일치율 전체 0.9446 (596행) | 급등영상 있는 채널 0.9456 (441행)


(b) 보정 K로 추정한 s_U: -0.25 | 로그우도 상위: {-0.25: -233.3, -0.2: -233.5, -0.3: -234.1}


### 6.3 결과
- (a) 공식 08-11 재현 일치율: **원값 0.985 > 보정값 0.945** → 공식 라벨은 원값(급등 포함)으로 계산됨. 9/1 past_score도 원값 → 원값 past_med 사용이 맞다.
- (b) t를 0.263으로 고정한 것이 문제: 라벨은 상대 기준(상위 20%)이라 K가 모두 부풀면 t도 같이 오른다 → **t와 9월 영상 수준 s를 동시 추정**해야 함.

**모든 기존 제출에 대한 함의**: 9/1 정답 = 원값 past(8월 말 급등 포함) vs 9월 영상. 08-26~31에 많이 올린 채널은 past가 부풀어 오히려 Rising이 어렵다. 그런데 원값 최근성 피처(mom_7, ew3)는 이들을 "급상승"으로 보고 높게 예측 → **방향이 반대**.

### 6.4 (t, s) 2차원 최대우도 (K 원값)

In [15]:
def mle_ts(df, dists, t_grid, s_grid, n=300, jitter=0.3, u_days=11, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    base = []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        if pd.isna(r.prev) or pd.isna(r.past_l_prev): continue
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0: continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku = rng.poisson(rate * u_days, n); mu_ = max(ku.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
        base.append((int(r.prev), K, U, r.past_l_prev))
    LL = np.zeros((len(t_grid), len(s_grid)))
    for j, s in enumerate(s_grid):
        meds = [(_nanmed_rows(np.hstack([np.tile(K, (U.shape[0], 1)), U + s]) if len(K) else U + s) - pp, y) for y, K, U, pp in base]
        for i, t in enumerate(t_grid):
            tot = 0.0
            for g, y in meds:
                p = np.clip((g >= t).mean(), 0.01, 0.99); tot += np.log(p if y == 1 else 1 - p)
            LL[i, j] = tot
    i, j = np.unravel_index(np.argmax(LL), LL.shape)
    return float(t_grid[i]), float(s_grid[j]), pd.DataFrame(LL, index=t_grid, columns=s_grid)

TG = np.round(np.arange(0.1, 1.31, 0.05), 2)
SG = np.round(np.arange(-0.6, 1.01, 0.1), 2)
# 점검: 학습 날짜 2개 (정답 t 알고 있음, s는 0이어야)
for D_ in ['2026-07-30', '2026-08-03']:
    d = pairs[pairs.ref_date == D_]
    th, sh, _ = mle_ts(d, chan_dist(D_), TG, SG)
    print(f'점검 {D_}: t̂={th}, ŝ={sh} (정답 t={d.t_prev_true.iloc[0]:.3f}, s=0)')
T_HAT, S_HAT, LLts = mle_ts(test, dist_test_adj, TG, SG)
print(f'테스트 08-25: t̂={T_HAT}, ŝ={S_HAT}')
best = LLts.stack().sort_values(ascending=False).head(6).round(1)
print('로그우도 상위 (t, s):', best.to_dict())

점검 2026-07-30: t̂=0.3, ŝ=0.1 (정답 t=0.277, s=0)


점검 2026-08-03: t̂=0.25, ŝ=0.1 (정답 t=0.252, s=0)


테스트 08-25: t̂=1.25, ŝ=1.0
로그우도 상위 (t, s): {(1.25, 1.0): -203.8, (1.2, 1.0): -206.3, (1.2, 0.9): -206.6, (1.15, 0.9): -209.8, (1.25, 0.9): -211.1, (1.15, 0.8): -211.9}
